In [ ]:
import pandas as pd
import numpy as np

# Task 1: Handling Missing Values (Cleaning Empty Cells)

### Approximately 5% of the values in most columns have been randomly replaced with NaN.

Perform the following operations:

- Load the dataset and calculate the exact total and percentage of missing values for every column in the dataset.
- For categorical columns missing data, apply Mode imputation or fill with a placeholder string (e.g., "Unknown").
- For numerical columns missing data, decide between Mean or Median imputation and apply your chosen method. Add a brief comment in your code explaining why you chose one over the other.
- Verify that no missing values remain in the dataset after your operations.

In [ ]:
ecommerce=pd.read_csv("drive/MyDrive/messy_ecommerce_data.csv")

In [ ]:
ecommerce.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50500 entries, 0 to 50499
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      50500 non-null  object 
 1   Customer_Age        48035 non-null  float64
 2   Transaction_Amount  48024 non-null  object 
 3   Transaction_Date    47944 non-null  object 
 4   Customer_Name       47973 non-null  object 
 5   Email_Address       47961 non-null  object 
 6   City                48014 non-null  object 
 7   Customer_Rating     48062 non-null  float64
 8   Is_Active           47945 non-null  object 
 9   Product_Category    47938 non-null  object 
dtypes: float64(2), object(8)
memory usage: 3.9+ MB


In [ ]:
ecommerce.describe()

,Customer_Age,Customer_Rating
count,48035.000000,48062.000000
mean,42.433600,4.067954
std,87.434567,9.875484
min,-18.000000,0.000000
25%,26.000000,2.000000
50%,35.000000,3.000000
75%,43.000000,4.000000
max,999.000000,99.000000


In [ ]:
ecommerce.isnull().sum()

,0
Transaction_ID,0
Customer_Age,2465
Transaction_Amount,2476
Transaction_Date,2556
Customer_Name,2527
Email_Address,2539
City,2486
Customer_Rating,2438
Is_Active,2555
Product_Category,2562


In [ ]:
clean_ecommerce=ecommerce.copy()

In [ ]:
clean_ecommerce=clean_ecommerce.dropna(subset='Customer_Name')

In [ ]:
missing = pd.DataFrame({"Missing Values": clean_ecommerce.isnull().sum(),
                        "Percentage Missing": (clean_ecommerce.isnull().sum()/len(clean_ecommerce)*100).round(2)})

In [ ]:
missing

,Missing Values,Percentage Missing
Transaction_ID,0,0.00
Customer_Age,2331,4.86
Transaction_Amount,2346,4.89
Transaction_Date,2422,5.05
Customer_Name,0,0.00
Email_Address,2409,5.02
City,2355,4.91
Customer_Rating,2299,4.79
Is_Active,2437,5.08
Product_Category,2443,5.09


In [ ]:
categorical_cols = clean_ecommerce.select_dtypes(include=["object"]).columns

In [ ]:
categorical_cols

Index(['Transaction_ID', 'Transaction_Amount', 'Transaction_Date',
       'Customer_Name', 'Email_Address', 'City', 'Is_Active',
       'Product_Category'],
      dtype='object')

In [ ]:
for col in categorical_cols:
    clean_ecommerce[col] = clean_ecommerce[col].fillna(clean_ecommerce[col].mode()[0])

In [ ]:
numerical_cols = clean_ecommerce.select_dtypes(include=["float64", "int64"]).columns

In [ ]:
# Median is used because it is less affected by outliers than the mean,
# making it a more robust choice for imputing missing numerical values.
for col in numerical_cols:
    clean_ecommerce[col] = clean_ecommerce[col].fillna(ecommerce[col].median())

In [ ]:
print("Leftover missing values:", clean_ecommerce.isnull().sum().sum())

Leftover missing values: 0


# Task 2: Standardizing Data Formats (Cleaning Wrong Format)

### Several columns contain data that is technically present but structurally incorrect for machine learning algorithms. Standardize these formats:

- Transaction_Date: The column features a mix of standard YYYY-MM-DD and non-standard DD/MM/YYYY formats, alongside invalid string inputs like "Not a date". Use pandas datetime functions to convert all valid entries into a uniform YYYY-MM-DD datetime object. Convert the invalid strings into NaT (Not a Time) and then drop those specific rows.
- Transaction_Amount: This column is currently being read as an object/string because it incorporates float values mixed with random `$` signs and "Error" text strings. Strip the `$` signs, replace the "Error" strings with NaN, and convert the entire column to a float64 datatype.
- Is_Active: This column has mixed booleans with string formats like "Yes" and "No". Map these strings to their proper boolean equivalents (True for "Yes", False for "No") so the column contains strictly boolean values.

In [ ]:
clean_ecommerce["Transaction_Date"] = pd.to_datetime(clean_ecommerce["Transaction_Date"], format="mixed", errors="coerce")

In [ ]:
clean_ecommerce.dropna(subset=["Transaction_Date"],inplace=True)

In [ ]:
ecommerce["Transaction_Amount"] = (ecommerce["Transaction_Amount"].str.replace("$", "", regex=False).replace("Error", np.nan)).astype(float)

In [ ]:
bool_mapping = {"Yes": True, "No": False}
clean_ecommerce["Is_Active"] = (clean_ecommerce["Is_Active"].replace(bool_mapping).astype(bool))

# Task 3: Correcting Invalid Entries (Cleaning Wrong Data)

### Some columns contain data of the correct type, but the values violate logical boundaries or formatting rules. Fix the following anomalies:

- `Customer_Age`: The dataset contains impossible biological ages, including negative numbers (-10) and extreme outliers (150, 999). Filter the dataset to either drop rows where the age is strictly outside the logical range of 18 to 100, or replace those specific outliers with the median age of the dataset.
- `Customer_Rating`: The rating is strictly a 1-5 scale, but includes out-of-bounds metrics like 0, 6, 10, and 99. Use a clipping function or conditional replacement to ensure no rating falls outside the 1.0 to 5.0 range.
- `City`: The categorical text introduces inconsistent casing (e.g., lowercase, UPPERCASE) and extra leading/trailing whitespaces. Apply string manipulation functions to strip all invisible whitespaces and convert all city names to standard Title Case (e.g., "new york " becomes "New York").

In [ ]:
median_age = clean_ecommerce.loc[(clean_ecommerce["Customer_Age"] >= 18) & (clean_ecommerce["Customer_Age"] <= 100), "Customer_Age"].median()
clean_ecommerce.loc[(clean_ecommerce["Customer_Age"] < 18) | (clean_ecommerce["Customer_Age"] > 100), "Customer_Age"] = median_age
clean_ecommerce['Customer_Age'] = clean_ecommerce['Customer_Age'].fillna(median_age)
clean_ecommerce["Customer_Age"] = clean_ecommerce["Customer_Age"].astype("int32")

In [ ]:
# Any value below 1.0 becomes 1.0, and any value above 5.0 becomes 5.0.
clean_ecommerce["Customer_Rating"] = clean_ecommerce["Customer_Rating"].clip(lower=1.0, upper=5.0)

In [ ]:
clean_ecommerce["City"] = (clean_ecommerce["City"].str.strip().str.title())

# Task 4: Removing Duplicates

### To prevent data leakage during model training and testing, redundant data must be removed.

- Identify how many exact duplicate rows exist in the dataset (Hint: there are exactly 500 scattered throughout).
- Drop these exact duplicate rows, keeping only the first occurrence of each observation.
- Print the final shape (number of rows and columns) of your cleaned dataframe.

In [ ]:
duplicate_count = clean_ecommerce.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)
clean_ecommerce = clean_ecommerce.drop_duplicates(keep="first")
print("Final Shape of Dataset:", clean_ecommerce.shape)
print("Remaining duplicate rows:", clean_ecommerce.duplicated().sum())

Number of duplicate rows: 442
Final Shape of Dataset: (44176, 10)
Remaining duplicate rows: 0


In [ ]:
clean_ecommerce.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44176 entries, 1 to 50499
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      44176 non-null  object        
 1   Customer_Age        44176 non-null  int32         
 2   Transaction_Amount  44176 non-null  object        
 3   Transaction_Date    44176 non-null  datetime64[ns]
 4   Customer_Name       44176 non-null  object        
 5   Email_Address       44176 non-null  object        
 6   City                44176 non-null  object        
 7   Customer_Rating     44176 non-null  float64       
 8   Is_Active           44176 non-null  bool          
 9   Product_Category    44176 non-null  object        
dtypes: bool(1), datetime64[ns](1), float64(1), int32(1), object(6)
memory usage: 3.2+ MB
